In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

/Users/romario/ML_from_scratch/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/romario/ML_from_scratch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = "ai_skepticism_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "shaistashahid/human-trust-levels-in-ai-systems",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df.head()

/var/folders/9b/2fjcrmgn6cn8c66zrqxjrw780000gn/T/ipykernel_5934/2212887.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,ai_model_name,query_category,ai_confidence_percentage,response_character_count,has_cited_sources,contains_hedging_words,includes_disclaimer,answer_detail_level,respondent_age_bracket,education_level,...,urgency_level,belief_alignment_status,subject_matter_expertise,trust_score_out_of_10,performed_fact_check,fact_check_method_used,verification_duration_mins,answer_accuracy_percentage,trust_calibration_valid,user_skepticism_category
0,Claude,math_calculation,93.23,527,True,False,False,Moderate,45-54,Bachelors,...,Low,False,False,10.0,True,Google Search,10.7,80.15,True,Moderate Trust
1,Llama,recipe_cooking,84.47,581,False,False,False,Specific,55-64,PhD,...,NaN,False,True,7.9,True,Asked Expert,44.3,92.33,True,Skeptical
2,Claude,general_knowledge,69.82,484,True,True,False,Very Specific,35-44,High School,...,NaN,True,False,8.6,True,Consulted Documentation,37.5,67.32,True,Moderate Trust
3,Claude,creative_writing,79.61,73,True,True,False,Specific,45-54,Professional,...,NaN,NaN,False,8.9,True,Checked Official Source,22.7,73.12,True,Moderate Trust
4,Claude,creative_writing,67.71,146,False,True,False,Vague,55-64,Masters,...,NaN,True,True,9.0,True,Academic Paper,43.7,81.05,True,Moderate Trust


In [3]:
df.info(), df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ai_model_name               1000 non-null   object 
 1   query_category              1000 non-null   object 
 2   ai_confidence_percentage    1000 non-null   float64
 3   response_character_count    1000 non-null   int64  
 4   has_cited_sources           1000 non-null   bool   
 5   contains_hedging_words      1000 non-null   bool   
 6   includes_disclaimer         1000 non-null   bool   
 7   answer_detail_level         1000 non-null   object 
 8   respondent_age_bracket      1000 non-null   object 
 9   education_level             1000 non-null   object 
 10  digital_literacy_score      1000 non-null   object 
 11  ai_familiarity_level        1000 non-null   object 
 12  decision_importance         1000 non-null   object 
 13  urgency_level               779 no

(None,
 ai_model_name                   0
 query_category                  0
 ai_confidence_percentage        0
 response_character_count        0
 has_cited_sources               0
 contains_hedging_words          0
 includes_disclaimer             0
 answer_detail_level             0
 respondent_age_bracket          0
 education_level                 0
 digital_literacy_score          0
 ai_familiarity_level            0
 decision_importance             0
 urgency_level                 221
 belief_alignment_status       342
 subject_matter_expertise        0
 trust_score_out_of_10           0
 performed_fact_check            0
 fact_check_method_used        372
 verification_duration_mins      0
 answer_accuracy_percentage      0
 trust_calibration_valid         0
 user_skepticism_category        0
 dtype: int64)

In [4]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')
imputer.fit_transform(df)

df.isnull().sum()


ai_model_name                   0
query_category                  0
ai_confidence_percentage        0
response_character_count        0
has_cited_sources               0
contains_hedging_words          0
includes_disclaimer             0
answer_detail_level             0
respondent_age_bracket          0
education_level                 0
digital_literacy_score          0
ai_familiarity_level            0
decision_importance             0
urgency_level                 221
belief_alignment_status       342
subject_matter_expertise        0
trust_score_out_of_10           0
performed_fact_check            0
fact_check_method_used        372
verification_duration_mins      0
answer_accuracy_percentage      0
trust_calibration_valid         0
user_skepticism_category        0
dtype: int64

In [5]:
cat_features = [feature for feature in df.columns if df[feature].dtype == 'object']
encoders = {}

for feature in cat_features:
    le = LabelEncoder()
    df[feature] = le.fit_transform(df[feature].astype(str))
    encoders[feature] = le

In [6]:
X, y = df.drop('user_skepticism_category', axis=1), df['user_skepticism_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
gini_cls = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    bootstrap=True,
    oob_score=True,
    criterion='gini')
gini_cls.fit(X_train, y_train)

mean_acc = gini_cls.score(X_test, y_test)

print("Mean Accuracy:", mean_acc)
print("OOB Score:", gini_cls.oob_score_)


Mean Accuracy: 0.99
OOB Score: 0.98875


In [8]:
entropy_cls = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    bootstrap=True,
    oob_score=True,
    criterion='entropy')
entropy_cls.fit(X_train, y_train)

mean_acc = entropy_cls.score(X_test, y_test)

print("Mean Accuracy:", mean_acc)
print("OOB Score:", entropy_cls.oob_score_)

Mean Accuracy: 0.99
OOB Score: 0.9875


In [9]:
log_cls = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    bootstrap=True,
    oob_score=True,
    criterion='log_loss')
log_cls.fit(X_train, y_train)

mean_acc = log_cls.score(X_test, y_test)

print("Mean Accuracy:", mean_acc)
print("OOB Score:", log_cls.oob_score_)

Mean Accuracy: 0.99
OOB Score: 0.9875
